## Counterfactual explanations: Practice

Among the libraries that let you implement Counterfactual explanations, the most popular one is [DiCE](https://github.com/interpretml/DiCE/tree/main). It is exactly with DiCE (Diverse Counterfactual Explanations) that we will work, since it is the most flexible to use. Explanations can be obtained for models trained with `sklearn, keras, tensorflow` and `pytorch`, but only for tabular data.

A slightly less known library is [CARLA](https://github.com/carla-recourse/CARLA). If you are solving a classification task, we recommend taking a closer look at it, since it is broader *in terms of the ways* of obtaining a counterfactual explanation.

In [ ]:
!pip install dice-ml pandas>2 -q

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

data = pd.read_csv('https://github.com/SadSabrina/explainable_AI_course/raw/refs/heads/main/data/fetch_california_housing.csv',
                   index_col=0)
data.head()

**A counterfactual explanation: what for?**

A counterfactual explanation is used in the local sense — that is, when the prediction of the model is explained for some particular object from the test data.

**Case:** let's consider the task of predicting house prices. You have delivered the model to the direct customer and you need to find out why a particular house $x[i]$ was valued at the price $y_i$? What minimal changes have to be made so that the prediction falls into the interval $[y_{j-1}, y_j]$?

**What does a counterfactual explanation give us in the case of regression?**

**Answer:** As one of the possible options, an observation $x$ such that $\hat{f}(x)$ belongs to a given range $[y_i, y_j]$.

So as not to go far from the concept we walked through in the example, let's continue working with our beautiful California and its no less wonderful houses! Note that our dataset contains only continuous features. That is good for us, since we can avoid going deep into data preprocessing. To avoid the preprocessing step altogether, we will use the ensemble algorithm `RandomForestRegressor`.

**Quiz 1:** Which other explanation methods among the ones we have already studied could we apply to this algorithm?

In [ ]:
#Let's prepare the data for training

X = data.drop('target', axis=1)
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from IPython.display import display

forest = RandomForestRegressor(random_state=42)
params = {'max_depth': [6, 10, 14]}

gs = GridSearchCV(forest, params, cv=3, scoring='neg_mean_squared_error')
gs.fit(X_train, y_train)

estimator = gs.best_estimator_

print('Cv score:')
display(pd.DataFrame(gs.cv_results_))
print('Test MSE:', mean_squared_error(estimator.predict(X_test), y_test))

##**DiCE**



To find the nearest counterfactual object, we have to define:
- by which criterion to search (the counterfactuality of the prediction)
- where to search
- how to search
- for whom to search

We will answer these questions using DiCE (both for classification and for regression) by defining objects of two classes: Data and Model. In case you want to preserve the privacy of the data, DiCE also provides this possibility (we will walk through it a bit below as well).

To use `Data` and `Model` you have to:

- save the names of the continuous and categorical features into the lists `continuous_features` and `discrete_features` respectively
- train the model

We formulate our query to the counterfactual explanation as:

`In the whole dataset (we will give it to you) find the nearest counterfactuals for the chosen instance, such that the prediction $\hat{f}(x)$ for it is $\in [y_i, y_j]$`

1. **Where to search** — we specify this in the object of the `Data` class. It can be any set of data whose structure is the same as that of the data the model was trained on.
2. **How to search** — here DiCE offers three search methods, `kdtree`, `genetic` and `random`. The counterfactuals that are found depend, among other things, on the chosen method. That is, a situation where you have found 3 counterfactuals and they are stable from one search method to another **will not happen**. Counterfactuals that are stable from iteration to iteration are given by `kdtree`, so we recommend using it.
3. **For whom to search** - any object from the test dataset.

**Where to search**

You can look for a counterfactual example both in the training and in the test dataset.

**The training dataset** \

The goal: debugging the model during training. \

**Advantages:**
- You can use all the available data for the analysis, since the training sample is the largest set of data available to us.
- You can better understand how the model learns and which features influence the predictions the most. In some cases this helps to remove undesirable objects or to correct the distributions of some features.

**Disadvantages:**
- Counterfactual examples may not reflect reality if the model is overfitted on the training data.
- The examples that are found may be not relevant for new, previously unseen data.

**The test dataset**

The goal: Evaluation and interpretation of the model on new, unseen data.

**Advantages:**
- It lets you check how the model will behave in real scenarios.

**Disadvantages:**
- A smaller amount of data for the analysis compared with the training set.

**General recommendations:**

- For research purposes and for interpreting the model during its development use the training dataset. This will help to better understand the internal working of the model and its reactions to changes of the features.
- For evaluating the model and its behaviour on new data: use the test dataset. This will help to check how the model will work in real conditions and how stable its predictions are to changes in the data.

We will consider the scenario of searching for a counterfactual object in the test data `X_test`. Although to define the `Data` class the set that is used has to fully satisfy the structure of the data the model was trained on, we do not necessarily have to include the values of the target variable into `test`. It is enough to put a "stub" on them with any value.

In [ ]:
sample = X_test.sample(n=20, random_state=42)
data_to_search = sample.copy()

X_train_dice = X_train.copy()
X_train_dice['target'] = y_train

In [ ]:
import dice_ml
from dice_ml import Dice

continuous_features= list(X.columns)

d_housing = dice_ml.Data(dataframe=X_train_dice, outcome_name='target', continuous_features=continuous_features)

m_housing = dice_ml.Model(model=estimator, backend="sklearn", model_type='regressor')

The search for a counterfactual explanation is implemented with an object of the `Dice` class. DiCE includes *3 search methods* to choose from: `genetic`, `kdtree`, `random`.

**What the search method affects:**
- the **speed** of the search — (I) `random`, (II) `genetic`, (III) `kdtree`, the time changes depending on the number of observations for which the counterfactual observations are looked for (it is not obligatory to search only for one data instance
- the **stability** of the search — the counterfactual object that is found does not have to be the only one or the best one, because of which the search methods can return different objects from iteration to iteration. The most stable one (checked on small data) is `kdtree`. We recommend using it.

In [ ]:
exp_genetic_housing = Dice(d_housing, m_housing, method="kdtree") #we initialize the object with which we will look for the counterfactual explanation

query_instance_housing = data_to_search[:1]

print('Model prediction:', """Your code here""")

In [ ]:
query_instance_housing

**Let's ask ourselves a question:** how do we change the object minimally so that the prediction of the model doubles?

**Quiz 2:** Set the search interval for this task in the form: $[y\_twice, y\_twice + 0.5]$. As the answer in the trainer write down the right boundary (round it to two decimal places).

In [ ]:
genetic_housing_twice = exp_genetic_housing.generate_counterfactuals(query_instance_housing,
                                                               total_CFs=3,
                                                               desired_range=#Your code here)
genetic_housing_twice.visualize_as_dataframe(show_only_changes=True)

**Quiz 3:** Set the search interval:  $[4, 5]$ \
As the answer enter the value of the Population variable for the objects that were found

In [ ]:
genetic_housing_twice_up = exp_genetic_housing.generate_counterfactuals(query_instance_housing,
                                                               total_CFs=3,
                                                               desired_range=# Your code here)
genetic_housing_twice_up.visualize_as_dataframe(show_only_changes=True)

**Which of the ones found is the nearest? The Euclidean metric**

The distance between the objects $x, y$ by definition is not at all $\sqrt{\sum_i^n (x_i - y_i)^2}$.

In mathematics a distance is a *metric* and is declared axiomatically. Do not be scared, you do not have to know the axioms. From this fact you need to understand that a distance is one of the ways to evaluate how far the objects are from each other.

The distance that is introduced at school ($p(x, y) = \sqrt{\sum_i^n (x_i - y_i)^2}$) is only one of the examples. But it can and should be used! It is called the Euclidean distance. If your tabular data reflects a specific structure, for example text (let's say in the form of Tf-idf encoding), then it can be more productive to consider another kind of distance.  

In [ ]:
#Let's save the objects found for the 2x interval as a list
cf_objects = genetic_housing_twice.cf_examples_list[0].final_cfs_df.values
cf_objects_4_5 = # your code here (the objects for task 4)

Let's compute the Euclidean distance between the explained example and the examples that were found. We will compute the distance over all the coordinates except the target one.

In [ ]:
def euclidean_distance(x, y):

  distance = np.sqrt(np.sum((x-y)**2))
  return distance


def cheb_distance(x, y):

  distance = np.max(np.abs((x-y)))
  return distance

In [ ]:
query_instance_housing_as_vector = query_instance_housing.values

**Quiz 4:** Compute the Euclidean distance between the counterfactual objects that were found and the explained example. What is the smallest one? Round the answer to two decimal places.

In [ ]:
for i in cf_objects:
  print('Euclidean dist:', #Your code here)

If all we care about is that the changes are the smallest in every coordinate, then we can compute the Chebyshev distance by the formula $p(x, y) = max(|x_i-y_i|)$

**Quiz 5:** Compute the Chebyshev distance for the same objects. Did the result change? Choose the answers that correspond to the conclusions from the values you got.

Carry out your own analysis for the object number 7, without changing the way the explanations are searched for. Look for the counterfactual objects in the interval $[model\_prediction*2, model\_prediction*2 + 0.5]$

In [ ]:
query_instance_housing_two = #Your code here
print('Model prediction', estimator.predict(query_instance_housing_two))

genetic_housing_seven = exp_genetic_housing.generate_counterfactuals(query_instance_housing_two,
                                                               total_CFs=3,
                                                               desired_range=#Your code here)

genetic_housing_seven.visualize_as_dataframe(show_only_changes=True)

**Quiz 6:** What is the largest target among the 3 objects that were found? Round the answer to two decimal places.

**Working in DiCE without access to the training data**

In fact, we have already walked through almost all of this step. DiCE is a method that depends more on the model and on how it was trained, so there is no need to show the training data. But some statistics of the data still have to be known, since the method needs to understand where to search.  Otherwise, unfortunately, you will not get any results from the method.

So, to work with anonymized data it is enough to list all the features and their possible values in the `features` attribute. Let's put together a simple example — we will predict the class of the Irises you already know, so as not to spend a long time retraining a new model.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

X, y = load_iris(return_X_y=True, as_frame=True)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=42)

model = DecisionTreeClassifier(max_depth=3)
model.fit(X_tr, y_tr)

print('Model accuracy:', accuracy_score(model.predict(X_te), y_te))

Let's look at the list of features and the ranges of their values.

In [ ]:
X_tr.describe()

We put the data together, specifying the boundaries of the numeric features. For the categorical ones we would have had to list the categories. The order of the features has to be the same as when the model was trained.

In [ ]:

d_without_access = dice_ml.Data(features={'sepal length (cm)': [4.3, 7.7],
                                          'sepal width (cm)': [2, 4.2],
                                          'petal length (cm)': [1.1, 6.7],
                                          'petal width (cm)' : [0.1, 2.5]},
                 outcome_name='target')

We declare the model class in practically the same way. Instead of a model trained directly during the notebook, you can also add the path to a saved pretrained model in the 'model_path' parameter.

In [ ]:
backend = 'sklearn'
m = dice_ml.Model(model=model, backend=backend, model_type='classifier')

query_instance = pd.DataFrame({'sepal length (cm)': 12,
                               'sepal width (cm)': 3,
                               'petal length (cm)': 1,
                               'petal width (cm)': 0.7},
                               index=[0])

We declare the "search" and get the results. However, the story without access to the data has a significant downside — it is impossible to search with the most stable kdtree.

In [ ]:
exp = dice_ml.Dice(d_without_access, m, method="genetic")

dice_exp = exp.generate_counterfactuals(query_instance, total_CFs=3, desired_class=2,  initialization="random"
                                       )

dice_exp.visualize_as_dataframe(show_only_changes=True)

**Quiz 7:** Did you manage to get stable counterfactual explanations?

## **Conclusions**
In this section we got to know the explanation of models by generating counterfactual explanations with DiCE. We found out that:
- the data sample on the basis of which the counterfactual values will be generated has to be formulated clearly
- the method does not always bring stable explanations
- distance metrics can be used to make the closeness more precise
- the method does not require direct access to the training data.